# Training Loop and Model Checkpointing

## Objectives
- Understand train() and eval() modes
- Build a complete training loop with validation
- Implement early stopping
- Learn model checkpointing best practices
- Monitor training metrics

## Introduction
A proper training loop is essential for deep learning projects. This notebook covers the complete workflow including validation, early stopping, and model persistence.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


## 1. Understanding train() and eval() Modes

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
# Create a simple model with Dropout and BatchNorm
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 2)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = DemoModel()
print(model)

# Key difference between modes
x = torch.randn(4, 10)

# Training mode
model.train()
print(f"\nTraining mode (model.training = {model.training}):")
# Iterate through batches of data
for name, module in model.named_modules():
    if hasattr(module, 'training'):
        print(f"  {name}: training={module.training}")

# Eval mode
model.eval()
print(f"\nEval mode (model.training = {model.training}):")
# Iterate through batches of data
for name, module in model.named_modules():
    if hasattr(module, 'training'):
        print(f"  {name}: training={module.training}")


## 2. Create Sample Dataset

## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
# Generate synthetic dataset
n_samples = 500
X = torch.randn(n_samples, 10)
y = (X[:, 0] + X[:, 1] > 0).long()

# Split train/val/test
train_size = int(0.6 * len(X))
val_size = int(0.2 * len(X))
test_size = len(X) - train_size - val_size

X_train, X_val, X_test = X[:train_size], X[train_size:train_size+val_size], X[train_size+val_size:]
y_train, y_val, y_test = y[:train_size], y[train_size:train_size+val_size], y[train_size+val_size:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# DataLoaders
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)


## 3. Training Function

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
def train_epoch(model, train_loader, optimizer, criterion, device):
# Iterate through batches of data
    """Train for one epoch"""
    model.train()
# Compute the loss (error) between predictions and actual values
    total_loss = 0
    correct = 0
    total = 0
    
# Iterate through batches of data
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass
        logits = model(X_batch)
# Compute the loss (error) between predictions and actual values
        loss = criterion(logits, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
# Compute the loss (error) between predictions and actual values
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    return total_loss / len(train_loader), correct / total

print("Training function defined.")


## 4. Validation Function

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
def validate(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
# Compute the loss (error) between predictions and actual values
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
# Iterate through batches of data
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
# Compute the loss (error) between predictions and actual values
            loss = criterion(logits, y_batch)
            
# Compute the loss (error) between predictions and actual values
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    return total_loss / len(val_loader), correct / total

print("Validation function defined.")


## 5. Early Stopping Class

## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Define a custom function with detailed implementation
class EarlyStopping:
    """Early stopping to avoid overfitting"""
    def __init__(self, patience=5, delta=0, verbose=True):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.counter = 0
# Compute the loss (error) between predictions and actual values
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
# Compute the loss (error) between predictions and actual values
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
# Compute the loss (error) between predictions and actual values
            self.best_loss = val_loss
            self.counter = 0

print("EarlyStopping class defined.")


## 6. Complete Training Loop with Checkpointing

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
def train_with_early_stopping(model, train_loader, val_loader, epochs=50, 
                               lr=0.001, patience=5, save_dir="checkpoints"):
    """
    Complete training loop with validation, early stopping, and checkpointing
    """
    # Setup
    save_dir = Path(save_dir)
    save_dir.mkdir(exist_ok=True)
    
# Update model parameters based on computed gradients
    optimizer = optim.Adam(model.parameters(), lr=lr)
# Compute the loss (error) between predictions and actual values
    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    # History
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    # Training
# Iterate through batches of data
    for epoch in range(epochs):
# Compute the loss (error) between predictions and actual values
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
# Compute the loss (error) between predictions and actual values
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Print progress
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
            print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        
        # Save checkpoint if best
# Compute the loss (error) between predictions and actual values
        if epoch == 0 or val_loss < min(history['val_loss'][:-1]):
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc
            }
            torch.save(checkpoint, save_dir / 'best_model.pt')
        
        # Early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    return history

print("Training function with early stopping defined.")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
# Train model
model = DemoModel().to(device)
history = train_with_early_stopping(
    model, train_loader, val_loader, 
    epochs=100, lr=0.001, patience=10,
    save_dir="./checkpoints"
)


## 7. Loading and Using Checkpoint

## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
# Load best model
checkpoint = torch.load('./checkpoints/best_model.pt')
model_loaded = DemoModel().to(device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['val_loss']:.4f}")
print(f"Best validation accuracy: {checkpoint['val_acc']:.4f}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
# Test final model
model_loaded.eval()
# Compute the loss (error) between predictions and actual values
test_loss = 0
correct = 0
total = 0
# Compute the loss (error) between predictions and actual values
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
# Iterate through batches of data
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        logits = model_loaded(X_batch)
# Compute the loss (error) between predictions and actual values
        loss = criterion(logits, y_batch)
        
# Compute the loss (error) between predictions and actual values
        test_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

test_acc = correct / total
# Compute the loss (error) between predictions and actual values
test_loss = test_loss / len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Configure loss function and optimization algorithm
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
# Compute the loss (error) between predictions and actual values
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
# Compute the loss (error) between predictions and actual values
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Configure loss function and optimization algorithm
# Summary statistics
print("\nTraining Summary:")
print(f"Best validation loss: {min(history['val_loss']):.4f}")
print(f"Best validation accuracy: {max(history['val_acc']):.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(f"\nTotal epochs trained: {len(history['train_loss'])}")
print(f"Final train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")


## Interview Questions & Answers

**Q1: What are the key takeaways from this section on Training Loop and Model Checkpointing?**

A: The key takeaways from this section include understanding how the fundamental concepts work together to enable deep learning. 
We've explored the interaction between different components and how they contribute to model training and performance. 
This foundation is essential for any deep learning practitioner because it explains the 'why' behind the 'how.' 
Real interview questions often explore whether candidates understand not just the mechanics but the reasoning behind design choices. 
When you can explain these concepts clearly and connect them to practical applications, you demonstrate genuine comprehension rather than rote memorization.

**Q2: How would you explain Training Loop and Model Checkpointing to a non-technical person?**

A: I would explain it using analogies to familiar concepts. Training Loop and Model Checkpointing works like [appropriate analogy], where the components interact similar to how [real-world example] works. 
The core idea is that systems learn by observing patterns and adjusting themselves based on feedback. This mirrors how humans learn – by trying things, seeing results, and adjusting our approach. 
Companies care about this explanation because they often need to communicate with stakeholders who aren't technically skilled, so the ability to explain complex concepts simply is highly valued.

**Q3: What common mistakes do people make when working with Training Loop and Model Checkpointing?**

A: Common mistakes include not understanding the underlying principles and blindly applying techniques without knowing why they work. 
Another frequent error is poor data preparation – garbage in, garbage out applies directly to deep learning. 
People also often tune hyperparameters randomly instead of methodically, wasting computational resources. 
They might not validate their models properly or confuse correlation with causation in their results. 
The best practitioners understand that deep learning is as much an art as a science, requiring intuition informed by solid theoretical grounding.
